In [2]:
### Social Media Analytics Project (used for Academic Full Search)
# To extract data from Twitter using the tweepy Python library, tweepy (4.3.0)
# Author: Tena Bao

# Instructions for execution
# Prepare authentication token
# Input start_time and end_time before script execution

# Version 1.0, Feb 08, 2022:
# Use key words to earch for tweets posted within a time range.
# Create csv file "TwitterFullData_YYYYMMDD.csv". 
# Add time delay to code when tweet total reached a certain number to avoid "TooManyRequests" error.
# Handle "TooManyRequests" error
# Version 1.1, total % 15000 == 0*, sleep_time = 600
# Version 1.2, add logging to "get_full_data_tweepy_log_YYYY-MM-DD_HH-MM.log"
# Version 2.0 (TBT). To use authentication method "OAuth 2.0 App-only" and new "keys.py", 

import tweepy
import keys
import datetime
import pandas as pd

import urllib.request
import os
import os.path

# For image download and csv file generation
import requests
import numpy as np

import time
from datetime import datetime, timedelta

import logging


In [3]:
# Logging File
log_file_name = datetime.now().strftime('get_full_data_tweepy_log_%Y-%m-%d_%H-%M')
logging.basicConfig(filename=log_file_name, format='%(asctime)s|%(levelname)s|%(message)s', level=logging.DEBUG)#encoding='utf-8'

# To check if folder or file exists
def check_path_exist(file_name):
    file_exist = os.path.exists(file_name)
    if (file_exist):            
        return True
    else:
        return False    
      
# To create a folder to save images
# C:/Users/Tena/socialMediaEnv/Twitter Images Download_YYYYMMDD
# YYYYMMDD is today's date
date_of_today = datetime.today().strftime('%Y%m%d')
parent_dir = 'C:/Users/Tena/socialMediaEnv/'
csv_file_name = 'TwitterFullData' + '_' + date_of_today + '.csv'

def create_folder (parent_dir, dir):
    path = os.path.join(parent_dir, dir)
    if(check_path_exist(dir)):
        logging.error("Folder \"" + dir + "\" exists!")
        logging.shutdown()
        raise Exception("Folder \"" + dir + "\" exists!")  
    else:
        os.mkdir(path)
        print("Folder '% s' created" % dir)
            
def check_csv (file_name):
    if (check_path_exist(file_name)):
        logging.error("Csv file \"" + file_name + "\" exists!")
        logging.shutdown()
        raise Exception("Csv file \"" + file_name + "\" exists!") 
            
check_csv(csv_file_name)
    
# Authenticating with Twitter Via Tweepy
# Creating and Configuring an OAuthHandler to Authenticate with Twitter
try:
    consumer_key = keys.consumer_key
    consumer_secret = keys.consumer_secret
    access_token = keys.access_token
    access_token_secret = keys.access_token_secret
except KeyError:
    #sys.stderr.write("Key/Token Variable not Set\n")
    sys.exit(1)
        
auth = tweepy.OAuthHandler(consumer_key, consumer_secret)
auth.set_access_token(access_token, access_token_secret)
api = tweepy.API(auth, wait_on_rate_limit=True, )

client = tweepy.Client(bearer_token = keys.bearer_token)

In [4]:
# Print search limits
data = api.rate_limit_status()

#print(data['resources']['statuses'])
#print(data['resources']['search'])

{'/search/tweets': {'limit': 180, 'remaining': 180, 'reset': 1644544259}}


In [5]:

#SQL1: search all tweets with key words
query = '(((vaccine OR pfizer OR moderna OR "Johnson and Johnson" OR "Johnson & Johnson" OR J&J) booster) OR #BoosterVaccine OR #VaccineBooster OR #PfizerBooster OR #ModernaBooster OR #JohnsonAndJohnsonBooster OR #Johnson&JohnsonBooster OR #J&JBooster)'

# To collect data for US Eastern Time period: '2021-12-17T00:00:01Z' - '2021-12-24T00:00:01Z'
# Coordinated Universal Time period: '2021-12-18T05:00:01Z' - '2021-12-24T05:00:01Z'
start_time = '2022-02-07T05:00:01Z'
end_time = '2022-02-08T05:00:01Z'

def get_start_end_time():
    date_format = '%Y-%m-%d'
    sub_time = 'T05:00:01Z'
    now = datetime.now()
    current_day = now.strftime(date_format)
    start_day = (now - timedelta(6)).strftime(date_format)

    start_time = start_day + sub_time
    end_time = current_day + sub_time    
    start_end_time = [start_time, end_time]
    
    return start_end_time

#start_end_time = get_start_end_time()
#start_time = start_end_time[0]
#end_time = start_end_time[1]

print('Extract Twitter data from ' + start_time + ' to ' + end_time + '.')
logging.info('Extract Twitter data from %s to %s.', start_time, end_time)

max_fetch_number = 100
flatten_limit = 100#maximum number of results to yield
sleep_time = 600 #sleep_time = 300, hit TooManyRequests error in loop
loop_sleep_time = 300

tweet_f = ['author_id','conversation_id','created_at','id','in_reply_to_user_id','public_metrics','referenced_tweets']
user_f = ['id','created_at','username','location','verified']
place_f = ['id','full_name']
media_f = ['media_key','type']


def get_all_tweets_data(first_query, nt):
# Client.search_all_tweets(query, *, user_auth=False, end_time, expansions, max_results, media_fields, next_token, place_fields, poll_fields, since_id, start_time, tweet_fields, until_id, user_fields)
# Query statement can be 1024 characters long for Academic Research access.

    rate_limit_error = True
    tweets_data = None
    if(first_query == False):           
        try:   
            tweets_data = client.search_all_tweets(query = query,#fullSearch

                                             tweet_fields = tweet_f,                                         place_fields = ['id', 'full_name'],
                                             user_fields = user_f, #expansions='author_id',

                                             next_token = [nt],
                                             start_time = start_time,
                                             end_time = end_time, 
                                             max_results = max_fetch_number) 
            rate_limit_error = False
            
        except tweepy.errors.TooManyRequests as e:
            print('This is a TooManyRequests error (get_all_tweets_data).')
            print('Sleeping after TooManyRequests error (get_all_tweets_data) at: ', datetime.now())
            logging.warning('Sleeping after TooManyRequests error (get_all_tweets_data).')
            time.sleep(sleep_time)             
            pass
        except tweepy.TweepyException as e:
            print('This is a TweepyException (get_all_tweets_data).')            
            print('Sleeping after TweepyException (get_all_tweets_data) at: ', datetime.now())
            logging.warning('Sleeping after TweepyException (get_all_tweets_data).')
            time.sleep(sleep_time)             
            pass
    else:
        tweets_data = client.search_all_tweets(query = query,#fullSearch

                                     tweet_fields = tweet_f,
                                     user_fields = user_f, 

                                     start_time = start_time,
                                     end_time = end_time, 
                                     max_results = max_fetch_number)
            
    # When a response is returned without a next_token value, it can be assumed that all results have been paged through.
    if(rate_limit_error == False):
        next_token = tweets_data.meta.get('next_token', '')#next_token = tweets_data.meta['next_token']
        #print('next_token is: ', next_token)

    #print('tweets_data: ', tweets_data.meta)    
       
    return tweets_data

Extract Twitter data from 2022-02-07T05:00:01Z to 2022-02-08T05:00:01Z.


In [6]:
# Get tweet by tweet id
# https://developer.twitter.com/en/docs/twitter-api/tweets/lookup/api-reference/get-tweets-id
def get_one_tweet(id):
    get_tweet = client.get_tweet(id, tweet_fields = tweet_f,                                 
                                 user_fields  = user_f, 
                                 media_fields = media_f,
                                 #expansions   = ['author_id'])
                                 expansions   = ['author_id','referenced_tweets.id','attachments.media_keys'])
    
    return get_tweet


In [7]:
# Returns count of Tweets for a specific period that match a search query.
total_counts_by_test_period = client.get_all_tweets_count(query = query, granularity = 'day',#fullSearch
                                                             start_time = start_time,
                                                             end_time = end_time)
#print('total_counts_by_test_period: ', total_counts_by_test_period)

Forbidden: 403 Forbidden

In [1]:
output = []

# Twitter allows 10 million Tweets per month for full search, set the cap to 10,000,000 to avoid dead lock
# Get a notification when cap_limit has been reached
cap_limit = 10000000

# To total number of tweets retrieved
counter_total = 0 
 
# New in Tweepy version 4.0. Paginator.flatten() flattens the data and iterates over each object.
# Set 'pagination_token' to the value of next_token for the next page of results
def use_paginator(first_query, pagination_token, counter_total):
    counter = 0
    paginator_data = None
    if(first_query == False):
        try:
            paginator_data = tweepy.Paginator( client.search_all_tweets, query,#fullSearch
                                   tweet_fields = tweet_f,
                                   place_fields = place_f,
                                   user_fields = user_f,                                
                                   expansions= ['author_id', 'referenced_tweets.id'],

                                   #Use pagination_token
                                   pagination_token = [pagination_token],  

                                   start_time = start_time,
                                   end_time = end_time,
                                   max_results = max_fetch_number).flatten(flatten_limit)#max_result = 10 to 100
                                   #fullSearch: max_result = 10 to 500
        except tweepy.errors.TooManyRequests as e:
            print('This is a TooManyRequests error (user_paginator).')
            print('Sleeping after TooManyRequests error (user_paginator) at: ', datetime.now())            
            logging.warning('Sleeping after TooManyRequests error (user_paginator).')
            time.sleep(sleep_time)             
            pass
        except tweepy.TweepyException as e:
            print('This is a TweepyException (user_paginator).')
            print('Sleeping after TweepyException (user_paginator) at: ', datetime.now())
            logging.warning('Sleeping after TweepyException (user_paginator).')
            time.sleep(sleep_time)             
            pass
            
    else:
        paginator_data = tweepy.Paginator( client.search_all_tweets, query,#fullSearch
                               tweet_fields = tweet_f,
                               place_fields = place_f,                              
                               user_fields = user_f,
                               expansions= ['author_id', 'referenced_tweets.id'],

                               start_time = start_time,
                               end_time = end_time,
                               max_results = max_fetch_number).flatten(limit = flatten_limit)#max_result = 10 to 100
        
    if(paginator_data != None): 
        #TooManyRequests: 429 Too Many Requests, 20220120, 20220126
        f = True
        while f == True: 
            try:
                for tweet in paginator_data:
                    author_id = tweet.author_id
                    conversation_id = tweet.conversation_id
                    created_at = tweet.created_at
                    twt_id = tweet.id
                    in_reply_to_user_id = tweet.in_reply_to_user_id

                    #{'retweet_count': 0, 'reply_count': 0, 'like_count': 0, 'quote_count': 0}
                    tweet_public_metrics = tweet.public_metrics

                    retweet_count = tweet_public_metrics['retweet_count']#retweet_count = tweet_public_metrics.get('retweet_count'), int
                    reply_count = tweet_public_metrics['reply_count']
                    like_count = tweet_public_metrics['like_count']
                    quote_count = tweet_public_metrics['quote_count']

                    text = tweet.text


                    # At this time, the only expansion available to endpoints that primarily return user objects is expansions = 'pinned_tweet_id'        
                    # Additional query is needed to get user data, but using it will exceed rate limets for recent search.
                    # Rate limets for recent search: 180 per User, 450 per App (requests per 15-minute window unless otherwise stated)
                    # Error code: 429, https://developer.twitter.com/en/support/twitter-api/error-troubleshooting
                    #get_user = client.get_user(id = author_id, 
                                               #user_fields = ['created_at', 'username', 'location', 'verified'],
                                               #expansions = ['pinned_tweet_id']) 
                                               #)                         
                    #username =  get_user.data.username        

                    # referenced_tweets.type, id(, id.author_id, conversation_id, created_at, public_metrics)
                    ref_twt_type = 'None'
                    ref_twt_id = 'None'

                    referenced_tweets = tweet.referenced_tweets
                    if referenced_tweets:
                        ref_twt_type = referenced_tweets[0].type
                        ref_twt_id = referenced_tweets[0].id
                    #else:
                        #print('No referenced tweets.')            

                    #attch_media_key = 'None'
                    #attch_types = 'None'

                    #try:
                        #media_info = tweet.includes.get('media')
                        #attch_media_key = media_info.media_key
                        #attch_types = media_info.type

                    #except AttributeError:
                        #continue

                    ai_prefix = 'ai_'
                    ci_prefix = 'ci_'
                    ti_prefix = 'ti_'

                    to_ui_prefix = 'to_ui_'
                    if(in_reply_to_user_id == None):
                        to_ui_prefix = ''

                    ref_ti_prefix = 'rf_ti_'
                    if(ref_twt_id == 'None'):
                        ref_ti_prefix = ''

                    record = {'author_id':ai_prefix+str(author_id), 'conversation_id':ci_prefix+str(conversation_id), 'twt_created_at':created_at, 'twt_id':ti_prefix+str(twt_id), 'in_reply_to_user_id':to_ui_prefix+str(in_reply_to_user_id),
                              'retweet_count':retweet_count, 'reply_count':reply_count, 'like_count':like_count, 'quote_count':quote_count, 'text':text,
                              'ref_twt_type':ref_twt_type, 'ref_twt_id':ref_ti_prefix+str(ref_twt_id)}
                              #'Media Key':attch_media_key, 'Media Type':attch_types}

                    #print(record)
                    output.append(record)
                    counter += 1
                    counter_total +=1
                    if(counter_total > cap_limit):   
                        logging.error('counter_total is greater than cap_limit')
                        raise Exception('counter_total is greater than cap_limit ' + str(cap_limit) + '!')

                    #print('counter: ', counter)
                    #print('counter_total: ', counter_total)
                    logging.info('counter_total: %s', counter_total)
                    
                f == False
                return counter_total
            except tweepy.errors.TooManyRequests as e:
                f == True
                print('This is a TooManyRequests error (for loop).')
                print('Sleeping after TooManyRequests error (for loop) at: ', datetime.now())            
                logging.warning('Sleeping after TweepyException (for loop).')
                time.sleep(loop_sleep_time)#Must sleep to avoid duplicate record issue.             
                pass
            except tweepy.TweepyException as e:
                f == True
                print('This is a TweepyException (for loop).')
                print('Sleeping after TweepyException (for loop) at: ', datetime.now())
                logging.warning('Sleeping after TweepyException (for loop).')
                time.sleep(loop_sleep_time)             
                pass
        
    #return counter_total

def search_by_page(first_query, next_token, flag, total):
    first_sleep = False
    # Get some latest records for testing purpose
    #while (flag and total < 30):        
    while flag:
        print('print next_token at loop start: ', next_token)
        
        # "TooManyRequests: 429 Too Many Requests"
        # Beware of the rate limits, program to wait for a few minutes before retrying makes sense.               
        if((total > 0) and (total % 15000 == 0)):
        #if((total >= 15000) and first_sleep == False):
            print('total before sleep: ', total)
            print('Go sleeping at: ', datetime.now()) 
            logging.info('Go sleeping.')
            time.sleep(sleep_time)
            #first_sleep = True                
          
        tweets_data = get_all_tweets_data(first_query, next_token)                  
        if (tweets_data == None):
            print('tweet_data is None due to TooManyRequests error or TweepyException')
            print('next_token after TooManyRequests error: ', next_token)
            logging.warning('tweet_data is None due to TooManyRequests error or TweepyException')
            logging.warning('next_token after TooManyRequests error: $s.', next_token)
            
        else:                 
            if first_query == False:
                pagination_token = next_token
            else:
                pagination_token = ''                
            
            #TooManyRequests: 429 Too Many Requests, 20220120
            new_total = use_paginator(first_query, pagination_token, total)
            if (new_total == total): 
                print('counter_total not increased due to error')
                logging.warning('counter_total not increased due to error')
            else:
                #print('total before use_paginator: ', total)
                #print('total after use_paginator: ', new_total)
                total = new_total
                next_token = tweets_data.meta.get('next_token', '')
                result_count = tweets_data.meta.get('result_count', 0)
                print('result_count in loop: ', result_count)

                if(next_token != '' and len(next_token)>0 and result_count > 0):
                    flag = True
                    print('flag is True!')
                else:
                    flag = False
                    print('flag is False!')

            # Set first_query to False after the initial enquery
            first_query = False
            #print('print next_token at loop end: ', next_token)
            logging.info('next_token at loop end is: %s', next_token)
                    
    else:
        print('Loop ends!')        
        
first_query = True
flag = True
next_token = ''

print('Starting Twitter data extraction! The start date/time is: ', datetime.now())
logging.info('Starting Twitter data extraction!')
search_by_page (first_query, next_token, flag, counter_total)   

# Save csv file    
df = pd.DataFrame(output)

# Set starting index
starting_index = 1   #22983, 43782, 64964, 81242, 103906
df.index = np.arange(starting_index, len(df)+starting_index)

# Fix characters encoding issue.
df.to_csv(csv_file_name, mode='a', header=True, encoding = 'utf_8_sig') 
print('"' + csv_file_name + '"' + ' generated successfully! The generation date/time is: ', datetime.now())
logging.info('"' + csv_file_name + '"' + ' generated successfully!')
logging.shutdown()

NameError: name 'datetime' is not defined